# B / Historical features and validation

Editable transforms, annual writes and independent checks. Existing full results contain 48,538,627 rows.

B_v2 includes annual descriptions, wet frequency and conditional intensity, and 26 historical numerical features. The fixed eight-year annual panel is empty; no nationwide trend follows. Wet runs/recency inspect only the past hour and carry censor flags. C fits station coding on training only. Fixed-seed independent checks include year boundaries.

All expensive switches default to False. Run All displays existing results without full Spark computation or overwriting original results. Execute top-to-bottom with Shift+Enter. Enable the relevant switch and rerun the entire section after changes. Full Spark work requires Python 3.11, Java 17, project dependencies and formal data. Use new output directories; hand off source, experiment configuration and metadata with changes. Modified features require matching inference logic and fresh checks; old validation cannot certify a changed version.

## 1. Paths and shared helpers

In [ ]:
from pathlib import Path
import sys, json, csv, time, calendar, collections, math, shutil
from datetime import datetime
from decimal import Decimal
from IPython.display import display, HTML, Image

# Set the project root for your own computer.
import os
PROJECT_ROOT = Path(os.environ["PROJECT_ROOT"]).resolve() if os.environ.get("PROJECT_ROOT") else next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "scripts/data_release.py").is_file()), None)
if PROJECT_ROOT is None: raise RuntimeError("Set PROJECT_ROOT or open this notebook inside the extracted project")
DATA_ROOT = Path(os.environ.get("PROJECT_DATA_ROOT", PROJECT_ROOT / ".runtime/frozen_root")).resolve()

# Jupyter may start from the notebook directory or the project root.
candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
BROOT = next((p for p in candidates if (p / 'src/b_common.py').is_file()), None)
if BROOT is None:
    BROOT = PROJECT_ROOT / 'B_delivery'
assert (BROOT / 'src/b_common.py').is_file(), 'Extract the B source package into the project root first.'
assert (PROJECT_ROOT / 'scripts/data_release.py').is_file(), 'Check PROJECT_ROOT.'
sys.path.insert(0, str(BROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from b_common import csv_write, dump, log, metadata, release, rows, start
root = DATA_ROOT if (DATA_ROOT / "manifests/release_manifest.json").is_file() else PROJECT_ROOT
d = release(root)

# Display summary CSV without requiring pandas.
def show_csv(path, limit=15):
    data = rows(path)
    if not data:
        print('Empty table:', path)
        return
    from html import escape
    names = list(data[0])
    header = ''.join('<th>'+escape(k)+'</th>' for k in names)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(r[k]))+'</td>' for k in names)+'</tr>' for r in data[:limit])
    display(HTML('<div style="overflow-x:auto"><table><tr>'+header+'</tr>'+body+'</table></div>'))
    print(f'Showing {min(limit,len(data))} / {len(data)} rows; source:{path}')

print('Project:', PROJECT_ROOT)
print('B package:', BROOT)
print('Python：', sys.version.split()[0])


## 2. Inspect existing validation

In [ ]:
EXISTING_FEATURES = BROOT / 'results/features_v2'
show_csv(EXISTING_FEATURES / 'yearly_validation.csv')
print((EXISTING_FEATURES / 'validation.json').read_text())
print((EXISTING_FEATURES / 'inference_adapter_validation.json').read_text())


## 3. Parameters

Default does not recompute. Set RECOMPUTE_FEATURES=True after changes. Full eight-year output needs about 1 GB plus Spark temporary space.

In [ ]:
RECOMPUTE_FEATURES = False
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
NEW_FEATURES = BROOT / 'results' / ('features_notebook_' + RUN_TAG)
spark = None
print('New features:', NEW_FEATURES)


## 4. Time helpers

In [ ]:
from datetime import datetime, timedelta, timezone
SGT = timezone(timedelta(hours=8))
def epoch(year,month=1,day=1,hour=0,minute=0):
    return int(datetime(year,month,day,hour,minute,tzinfo=SGT).timestamp())


## 5. Editable production feature transform

This function actually participates in later computation. rangeBetween uses seconds; filter anchor year after window construction. Rerun after modifying windows/columns.

In [ ]:
def make_features(obs, year):
    """Causal features. Run length and recency are bounded by a complete 60m history."""
    from pyspark.sql import functions as F, Window
    begin, end = epoch(year), epoch(year + 1)
    x = (obs.filter(F.col('year').isin(year - 1, year))
        .filter(F.col('slot_end_epoch_s').between(begin - 3300, end - 1))
        .select('station_id', F.col('slot_end_epoch_s').alias('prediction_epoch_s'),
            F.col('slot_end_ts').alias('prediction_ts'), F.col('rain_mm').alias('current_5m_mm')))
    base = Window.partitionBy('station_id').orderBy('prediction_epoch_s')
    for label, left, right, count in [
        ('15m', -600, 0, 3), ('30m', -1500, 0, 6),
        ('60m', -3300, 0, 12), ('previous_15m', -1500, -900, 3)
    ]:
        w = base.rangeBetween(left, right)
        x = x.withColumn('n_valid_' + label, F.count('current_5m_mm').over(w))
        x = x.withColumn('rain_sum_' + label + '_mm',
            F.when(F.col('n_valid_' + label) == count, F.sum('current_5m_mm').over(w)))
    w60 = base.rangeBetween(-3300, 0)
    complete = F.col('n_valid_60m') == 12
    x = (x.withColumn('history_complete_60m', complete)
        .withColumn('rain_max_60m_mm', F.when(complete, F.max('current_5m_mm').over(w60)))
        .withColumn('rain_std_60m_mm', F.when(complete, F.stddev_pop('current_5m_mm').over(w60)))
        .withColumn('wet_slots_60m', F.when(complete,
            F.sum((F.col('current_5m_mm') > 0).cast('int')).over(w60)))
        .withColumn('rain_change_15m_mm', F.col('rain_sum_15m_mm') - F.col('rain_sum_previous_15m_mm')))
    # Exact offsets, not preceding-row positions: gaps remain unknown.
    for minutes in [5, 10, 15, 30]:
        w = base.rangeBetween(-60 * minutes, -60 * minutes)
        x = x.withColumn(f'lag_{minutes}m_mm',
            F.when(F.count('current_5m_mm').over(w) == 1, F.sum('current_5m_mm').over(w)))
    x = x.withColumn('wet_current', (F.col('current_5m_mm') > 0).cast('int'))
    for minutes in [5, 10]:
        x = x.withColumn(f'wet_lag_{minutes}m', (F.col(f'lag_{minutes}m_mm') > 0).cast('int'))
    x = (x.withColumn('rain_change_5m_mm', F.col('current_5m_mm') - F.col('lag_5m_mm'))
        .withColumn('rain_ratio_15m_30m',
            F.when(F.col('n_valid_30m') == 6,
                F.when(F.col('rain_sum_30m_mm') > 0,
                    F.col('rain_sum_15m_mm') / F.col('rain_sum_30m_mm')).otherwise(F.lit(0.0)))))
    x = (x.withColumn('_last_dry_epoch', F.max(F.when(F.col('current_5m_mm') == 0,
            F.col('prediction_epoch_s'))).over(w60))
        .withColumn('_last_wet_epoch', F.max(F.when(F.col('current_5m_mm') > 0,
            F.col('prediction_epoch_s'))).over(w60)))
    x = (x.withColumn('consecutive_wet_slots_60m', F.when(complete,
            F.when(F.col('_last_dry_epoch').isNull(), F.lit(12))
             .otherwise(((F.col('prediction_epoch_s') - F.col('_last_dry_epoch')) / 300).cast('int'))))
        .withColumn('wet_run_left_censored_60m', F.when(complete,
            F.col('_last_dry_epoch').isNull().cast('int')))
        .withColumn('no_rain_observed_60m', F.when(complete,
            F.col('_last_wet_epoch').isNull().cast('int')))
        # Raw recency remains null when no wet slot is observed; the exact age is unknown.
        .withColumn('minutes_since_last_rain_60m', F.when(complete & F.col('_last_wet_epoch').isNotNull(),
            (F.col('prediction_epoch_s') - F.col('_last_wet_epoch')) / 60.0))
        # 60 is a documented censoring code, accompanied by no_rain_observed_60m.
        # It does not assert that the last rainfall ended exactly 60 minutes ago.
        .withColumn('minutes_since_last_rain_60m_capped', F.when(complete,
            F.coalesce(F.col('minutes_since_last_rain_60m'), F.lit(60.0))))
        .drop('_last_dry_epoch', '_last_wet_epoch'))
    clock = F.hour('prediction_ts') + F.minute('prediction_ts') / F.lit(60.0)
    for name, angle in [('hour', 2 * math.pi * clock / 24.0),
                        ('month', 2 * math.pi * (F.month('prediction_ts') - 1) / 12.0)]:
        x = x.withColumn(name + '_sin', F.sin(angle)).withColumn(name + '_cos', F.cos(angle))
    return x.filter(F.col('prediction_epoch_s').between(begin, end - 1))


## 6. Independent timestamp verification

Python dictionary lookup reconstructs exact positions rather than calling the same Spark window logic. Update checks after feature changes, not just names.

In [ ]:
def validate_probes(spark, obs, feature_root, out):
    """Independent Python lookup for boundary cases + seeded 20-station/400-anchor probes."""
    import hashlib, random
    from collections import defaultdict
    from pyspark.sql import functions as F
    seed = 20261011
    boundary_stations = ['S24', 'S77', 'S113']
    anchors = [epoch(y, m, d, h, mi) for y in range(2017, 2025)
        for m, d, h, mi in [(1,1,0,0),(1,1,0,30),(5,15,14,30),(9,30,23,55),(12,31,23,55)]]
    boundary_keys = {(s, t) for s in boundary_stations for t in anchors}
    station_ids = sorted(r['station_id'] for r in obs.groupBy('station_id').count().filter(F.col('count') >= 20).select('station_id').collect())
    stations = random.Random(seed).sample(station_ids, min(20, len(station_ids)))
    sampled = (obs.filter(F.col('station_id').isin(stations))
        .select('station_id', 'slot_end_epoch_s')
        .filter(F.pmod(F.xxhash64('station_id', 'slot_end_epoch_s', F.lit(seed)), F.lit(1000)) == 0)
        .collect())
    per_station = defaultdict(list)
    for r in sampled:
        per_station[r['station_id']].append(r['slot_end_epoch_s'])
    random_keys = set()
    for s in stations:
        times = sorted(per_station[s], key=lambda t: hashlib.sha256(f'{seed}|{s}|{t}'.encode()).digest())
        # Sparse stations may need an exact hash-ordered fallback to supply 20 anchors.
        if len(times) < 20:
            times = [r['slot_end_epoch_s'] for r in obs.filter(F.col('station_id') == s)
                .select('slot_end_epoch_s').orderBy(F.xxhash64('slot_end_epoch_s', F.lit(seed))).limit(20).collect()]
        random_keys.update((s, t) for t in times[:20])
    keys = sorted(boundary_keys | random_keys)
    key_frame = spark.createDataFrame(keys, 'station_id string, prediction_epoch_s long')
    needed = sorted({(s, t + offset) for s, t in keys for offset in range(-3300, 1, 300)})
    need_frame = spark.createDataFrame(needed, 'station_id string, slot_end_epoch_s long')
    source = obs.select('station_id', 'slot_end_epoch_s', 'rain_mm').join(
        F.broadcast(need_frame), ['station_id', 'slot_end_epoch_s'], 'inner')
    lookup = {(r['station_id'], r['slot_end_epoch_s']): r['rain_mm'] for r in source.collect()}
    features = spark.read.parquet(str(feature_root))
    probes = features.join(F.broadcast(key_frame), ['station_id', 'prediction_epoch_s'], 'inner').collect()
    assert len(random_keys) == 400, 'Expected 20 eligible stations with 20 distinct anchors each'
    assert len(probes) > 400, 'Expected boundary probes plus 400 random probes'
    def equal(actual, expected, label):
        assert (actual is None) == (expected is None), (label, actual, expected)
        if expected is not None:
            assert math.isclose(actual, expected, rel_tol=1e-10, abs_tol=1e-8), (label, actual, expected)
    checked = []
    for row in probes:
        r = row.asDict(); s, t = r['station_id'], r['prediction_epoch_s']
        for label, offsets in [('15m', range(-600,1,300)), ('30m', range(-1500,1,300)),
                               ('60m', range(-3300,1,300)), ('previous_15m', range(-1500,-899,300))]:
            values = [lookup.get((s, t + offset)) for offset in offsets]
            known = [v for v in values if v is not None]
            assert r['n_valid_' + label] == len(known), (s, t, label)
            equal(r['rain_sum_' + label + '_mm'], sum(known) if len(known) == len(values) else None, label)
        values = [lookup.get((s, t + offset)) for offset in range(-3300,1,300)]
        complete = all(v is not None for v in values)
        assert r['history_complete_60m'] == complete
        for minutes in [5,10,15,30]:
            equal(r[f'lag_{minutes}m_mm'], lookup.get((s,t-minutes*60)), f'lag{minutes}')
        equal(r['wet_current'], int(lookup[(s,t)] > 0), 'wet_current')
        for minutes in [5,10]:
            v = lookup.get((s,t-minutes*60))
            equal(r[f'wet_lag_{minutes}m'], int(v > 0) if v is not None else None, 'wet_lag')
        lag = lookup.get((s,t-300))
        equal(r['rain_change_5m_mm'], lookup[(s,t)] - lag if lag is not None else None, 'change5')
        last30 = [lookup.get((s,t+o)) for o in range(-1500,1,300)]
        ratio = None
        if all(v is not None for v in last30):
            ratio = sum(last30[-3:])/sum(last30) if sum(last30) > 0 else 0.0
        equal(r['rain_ratio_15m_30m'], ratio, 'ratio')
        if complete:
            mean = sum(values)/12
            equal(r['rain_max_60m_mm'], max(values), 'max')
            equal(r['rain_std_60m_mm'], math.sqrt(sum((v-mean)**2 for v in values)/12), 'std')
            equal(r['wet_slots_60m'], sum(v > 0 for v in values), 'wet count')
            run = 0
            for v in reversed(values):
                if v <= 0: break
                run += 1
            ages = [i*5 for i,v in enumerate(reversed(values)) if v > 0]
            age = min(ages) if ages else None
            equal(r['consecutive_wet_slots_60m'], run, 'run')
            equal(r['wet_run_left_censored_60m'], int(run == 12), 'left censor')
            equal(r['no_rain_observed_60m'], int(not ages), 'no rain')
            equal(r['minutes_since_last_rain_60m'], age, 'raw age')
            equal(r['minutes_since_last_rain_60m_capped'], age if age is not None else 60, 'capped age')
        else:
            for column in ['rain_max_60m_mm','rain_std_60m_mm','wet_slots_60m','consecutive_wet_slots_60m',
                           'wet_run_left_censored_60m','no_rain_observed_60m',
                           'minutes_since_last_rain_60m','minutes_since_last_rain_60m_capped']:
                equal(r[column], None, column)
        checked.append({'station_id':s, 'prediction_epoch_s':t,
            'history_complete_60m':complete,
            'probe_type':'boundary' if (s,t) in boundary_keys else 'seeded_random', 'status':'PASS'})
    csv_write(out/'independent_timestamp_probes.csv', sorted(checked,key=lambda r:(r['station_id'],r['prediction_epoch_s'])))
    dump(out/'probe_design.json', {'seed':seed, 'random_stations':stations,
        'random_anchor_count':len(random_keys), 'boundary_keys_requested':len(boundary_keys),
        'actual_checked':len(probes), 'selection':'fixed seeded stations; stable hash sample and per-station hash order'})
    return len(probes)


## 7. Spark startup and formal data integrity

In [ ]:
if RECOMPUTE_FEATURES:
    if root == PROJECT_ROOT: raise RuntimeError("Prepare the frozen runtime first, or set PROJECT_DATA_ROOT to the verified frozen release")
    assert sys.version_info[:2] == (3,11), 'Use the project .venv Python 3.11 kernel.'
    begin = time.monotonic()
    out = NEW_FEATURES
    out.mkdir(parents=True, exist_ok=False)
    verification = d.verify_release(root, 'full')
    spark = start(root, 'B notebook historical features')
else:
    print("Skipped. To rerun, set RECOMPUTE_FEATURES to True in the parameters cell, then execute the section in order.")


## 8. Read source tables and annual reference counts

In [ ]:
if RECOMPUTE_FEATURES:
    from pyspark.sql import functions as F
    tables=d.read_tables(spark,root,'full',verify=False)
    obs=tables['observations']
    log('Reading reference yearly row counts and history-completeness counts')
    references={r['year']:r.asDict() for r in tables['targets'].groupBy('year').agg(
        F.count('*').alias('rows'),F.sum(F.col('historical_valid').cast('long')).alias('history_complete')).collect()}
    feature_root=out/'parquet'
    summaries=[]
else:
    print("Skipped. To rerun, set RECOMPUTE_FEATURES to True in the parameters cell, then execute the section in order.")


## 9. Annual feature generation/write/readback

Writes all eight years. Avoid another full Spark notebook during computation.

In [ ]:
if RECOMPUTE_FEATURES:
    for year in range(2017,2025):
        log(f'{year}: computing time windows with previous-year context')
        f=make_features(obs,year)
        target=feature_root/f'year={year}'
        f.write.mode('errorifexists').parquet(str(target))
        log(f'{year}: checking saved feature counts and null semantics')
        saved=spark.read.parquet(str(target))
        expressions=[F.count('*').alias('rows'),F.sum(F.col('history_complete_60m').cast('long')).alias('history_complete')]
        for label,n in [('15m',3),('30m',6),('60m',12),('previous_15m',3)]:
            invalid=((F.col('n_valid_'+label)==n)!=F.col('rain_sum_'+label+'_mm').isNotNull())
            expressions.append(F.sum(invalid.cast('long')).alias('invalid_'+label))
        from B_feature_api import NUMERIC_FEATURES
        from functools import reduce
        any_model_null = reduce(lambda left,right:left|right,
            (F.col(c).isNull() for c in NUMERIC_FEATURES))
        expressions.append(F.sum((F.col('history_complete_60m') & any_model_null).cast('long')).alias('invalid_model_nulls_complete'))
        invalid_age = F.col('history_complete_60m') & (
            F.col('minutes_since_last_rain_60m').isNull() != (F.col('no_rain_observed_60m') == 1))
        expressions.append(F.sum(invalid_age.cast('long')).alias('invalid_recency_semantics'))
        stats=saved.agg(*expressions).first().asDict()
        assert stats['invalid_model_nulls_complete']==0 and stats['invalid_recency_semantics']==0, stats
        assert stats['rows']==references[year]['rows'],(year,stats,references[year])
        assert stats['history_complete']==references[year]['history_complete'],(year,stats,references[year])
        assert all(stats['invalid_'+label]==0 for label in ['15m','30m','60m','previous_15m']),stats
        summaries.append({'year':year,**stats,'status':'PASS'})
        csv_write(out/'yearly_validation.csv',summaries)
        dump(out/'feature_schema.json',saved.schema.jsonValue())
        log(f'{year}: PASS, {stats["rows"]:,} anchors')
else:
    print("Skipped. To rerun, set RECOMPUTE_FEATURES to True in the parameters cell, then execute the section in order.")


## 10. Independent checks, prohibited-field checks and metadata

In [ ]:
if RECOMPUTE_FEATURES:
    log('Validating independent timestamp probes, including year boundaries')
    n_probes=validate_probes(spark,obs,feature_root,out)
    columns=spark.read.parquet(str(feature_root)).columns
    prohibited=['future_30m_mm','target_valid','model_anchor_eligible','fold1_role','fold2_role','fold3_role','final_role','year_boundary_safe']
    assert not set(columns).intersection(prohibited)
    dump(out/'validation.json',{'status':'PASS','rows':sum(r['rows'] for r in summaries),
        'history_complete':sum(r['history_complete'] for r in summaries),'independent_probes':n_probes,
        'yearly_counts_match_A_targets':True,'window_null_semantics':True,'no_future_columns':True,
        'key_uniqueness':'preserved from verified observations; windows neither duplicate nor group anchors',
        'input_integrity':verification,'feature_set_version':'B_v2',
        'complete_history_model_inputs_nonnull':True,'recency_censor_semantics':True})
    metadata(root,out,'python src/B_build_features.py --root PROJECT_ROOT',time.monotonic()-begin, {'feature_set_version':'B_v2'})
    log('All historical features PASS')
else:
    print("Skipped. To rerun, set RECOMPUTE_FEATURES to True in the parameters cell, then execute the section in order.")


## 11. Stop Spark

In [ ]:
if spark is not None:
    spark.stop()
    spark = None
    print('Spark stopped.')


## 12. Export modified feature functions

Offline generation and C inference must share transforms. Export to the new result directory, then hand off code/features to C and synchronize its predictor.

In [ ]:
if RECOMPUTE_FEATURES:
    import inspect
    original = (BROOT / 'src/B_build_features.py').read_text()
    import ast, textwrap
    for name, function in [('make_features', make_features), ('validate_probes', validate_probes)]:
        tree = ast.parse(original)
        node = next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name==name)
        lines = original.splitlines(keepends=True)
        lines[node.lineno-1:node.end_lineno] = [textwrap.dedent(inspect.getsource(function)).rstrip()+'\n']
        original = ''.join(lines)
    ast.parse(original)
    (NEW_FEATURES / 'B_build_features_notebook_version.py').write_text(original)
    print('Current functions exported:', NEW_FEATURES / 'B_build_features_notebook_version.py')
else:
    print("Skipped. To rerun, set RECOMPUTE_FEATURES to True in the parameters cell, then execute the section in order.")


## 13. Record modifications

In [ ]:
if RECOMPUTE_FEATURES:
    experiment = json.loads((BROOT / 'configs/B_experiment.json').read_text())
    experiment['notebook'] = 'B_Historical_Features.ipynb'
    experiment['changes'] = 'Describe window/column changes accurately here and update the features configuration.'
    dump(NEW_FEATURES / 'notebook_experiment.json', experiment)
    meta = json.loads((NEW_FEATURES / 'result_metadata.json').read_text())
    meta['experiment_config_file'] = 'notebook_experiment.json'
    meta['experiment_config_hash'] = d.sha256(NEW_FEATURES / 'notebook_experiment.json')
    meta['notebook_exported_code_hash'] = d.sha256(NEW_FEATURES / 'B_build_features_notebook_version.py')
    meta['outputs'] = [{'path':p.relative_to(NEW_FEATURES).as_posix(), 'bytes':p.stat().st_size, 'sha256':d.sha256(p)}
                       for p in sorted(NEW_FEATURES.rglob('*')) if p.is_file() and p.name!='result_metadata.json' and not p.name.startswith('.')]
    dump(NEW_FEATURES / 'result_metadata.json', meta)
    print('New features and change record:', NEW_FEATURES)
else:
    print("Skipped. To rerun, set RECOMPUTE_FEATURES to True in the parameters cell, then execute the section in order.")


# B-to-C feature handoff

Use features_v2 or the verified features_rerun_20261012 result. There are 48,538,627 anchors, keyed uniquely by station_id+prediction_epoch_s. Read observations/targets through the frozen release helper; map observation slot_end_epoch_s to prediction_epoch_s. Never join on station/date alone.

Filter C's actual fold role and joint past/future eligibility, apply training station support and deterministic training sampling, then audit the left anchor-to-feature join before threshold expansion. All left anchors must match once with no row inflation; unused extra right-side feature anchors are permitted. After expansion require unique station_id+prediction_epoch_s+threshold_mm and exactly the supported threshold count per anchor. The base-table audit alone is not a substitute for checking C's filtered/sampled inputs.

Model inputs are the 26 explicitly listed past-only numeric fields, numeric threshold and training-only station coding. Whole target records, future truth, completeness/eligibility and role fields cannot be passed into features. Inference requires supported station/threshold, five-minute grid and all 12 past slots, without requiring future truth. Definitions/time assumptions and censor flags are in docs/. C now completed E2 validation, locked selection, final 2024 evaluation and reload checks; see C_delivery and integrated report for actual scores.


## Supplement: time assumptions and evidence scope

The interval interpretation is a project assumption, not independently proven official semantics. Completed C handoff checks are separately recorded.

# Prediction time and rainfall interval contract

## Project assumptions and evidence boundary

The frozen project selects L1/D1: an aligned slot_end_ts is the endpoint of a five-minute interval. Original event_ts remains available; only 2017 epochs with remainder 299 modulo 300 are advanced by one second. This interval meaning is a disclosed engineering assumption, not an independently verified official definition.

The anchor is t = prediction_epoch_s = slot_end_epoch_s, in Asia/Singapore. Task 1 groups by rain_period_start_ts = slot_end_ts - 300 seconds. A midnight-ending interval starts at 23:55 on the preceding date.

## Last observation available at t

Under retrospective event-time availability, R(t) over (t-5min,t] is completed and available at t. No input endpoint may exceed t. Twelve exact endpoints t-55min,...,t cover (t-60min,t]. Publication/update delay is not reconstructed as an as-of historical version. If R(t) is unavailable at real prediction time, the cutoff, features and evaluation must be redesigned; the current result is not a demonstrated live-deployment result.

## Future labels

| Slot | Endpoint | Rainfall interval |
|---|---|---|
| 1 | t+5min | (t,t+5min] |
| 2 | t+10min | (t+5min,t+10min] |
| 3 | t+15min | (t+10min,t+15min] |
| 4 | t+20min | (t+15min,t+20min] |
| 5 | t+25min | (t+20min,t+25min] |
| 6 | t+30min | (t+25min,t+30min] |

future_30m_mm = R(t+300)+...+R(t+1800), covering (t,t+30min] and excluding R(t). All six positions are required, otherwise the total is null. Decimal totals are compared strictly with threshold_mm: equality is negative; unknown is not negative.

At 2019-05-15 14:30 SGT, the latest input interval is (14:25,14:30]; the future spans (14:30,14:35] through (14:55,15:00]. Use exact timestamp positions, not the previous 12/next 6 rows, and load cross-year context.

## Implementation status

A's frozen labels and B's offline/past-only features use this endpoint/six-slot contract. Cross-year, missing-slot and all-26-feature checks pass. C's actual post-sample joins, strict labels and saved-model past-only prediction interface now also pass. This verifies implementation consistency; official timestamp meaning and real publication availability remain assumptions to review before live deployment.


# Default numeric feature definitions (B_v2)

R(t) is rainfall over (t-5min,t] under the project L1/D1 assumption, not independently verified official semantics. Original event_ts is retained; prediction keys use aligned slot_end_epoch_s. Exact endpoints are matched by epoch seconds and missing positions are never zero-filled.

The following 26 rows follow B_feature_api.NUMERIC_FEATURES order. Station ID and numeric threshold are additional C inputs.

| Name | Definition/formula | Unit | Exact historical endpoints | Includes current slot | Missing/special values |
|---|---|---|---|---|---|
| current_5m_mm | Rainfall in the last completed five-minute interval | mm | Endpoint t; interval (t-5min,t] | yes | An observed valid current anchor is required. |
| lag_5m_mm | Rainfall at the exact endpoint t-5min | mm | Endpoint t-5min and its preceding five-minute interval | no | Null if the exact endpoint is absent; do not substitute the preceding row. |
| rain_sum_15m_mm | Sum over the complete past 15-minute interval | mm | Exact endpoints through t; interval (t-15min,t] | yes | All 3 exact endpoints are required; otherwise null, never a partial sum. |
| rain_sum_30m_mm | Sum over the complete past 30-minute interval | mm | Exact endpoints through t; interval (t-30min,t] | yes | All 6 exact endpoints are required; otherwise null, never a partial sum. |
| rain_sum_60m_mm | Sum over the complete past 60-minute interval | mm | Exact endpoints through t; interval (t-60min,t] | yes | All 12 exact endpoints are required; otherwise null, never a partial sum. |
| rain_sum_previous_15m_mm | Sum over the complete previous 15-minute interval | mm | t-25min,t-20min,t-15min; interval (t-30min,t-15min] | no | All 3 exact endpoints are required; otherwise null, never a partial sum. |
| rain_change_15m_mm | Past 15-minute sum minus the preceding 15-minute sum; not physical acceleration | mm | Two adjacent 15-minute windows through t | yes | Null if either operand is missing. |
| rain_max_60m_mm | Maximum five-minute rainfall in the complete past hour | mm | 12 exact endpoints t-55min,...,t | yes | Null unless the required exact historical slots exist; never replace missing rainfall with zero. |
| rain_std_60m_mm | Population standard deviation of the 12 past-hour rainfall values (divide by 12) | mm | 12 exact endpoints t-55min,...,t | yes | Null unless the required exact historical slots exist; never replace missing rainfall with zero. |
| wet_slots_60m | Number of positive-rainfall slots in the complete past hour, from 0 to 12 | slots | 12 exact endpoints t-55min,...,t | yes | Null unless the required exact historical slots exist; never replace missing rainfall with zero. |
| hour_sin | sin(2*pi*h/24); h is Singapore hour + minute/60, m is month 1-12 | dimensionless | Singapore calendar at prediction time t; no rainfall lookup | not applicable (time only) | Computed for a valid anchor time; independent of missing rainfall history. |
| hour_cos | cos(2*pi*h/24); h is Singapore hour + minute/60, m is month 1-12 | dimensionless | Singapore calendar at prediction time t; no rainfall lookup | not applicable (time only) | Computed for a valid anchor time; independent of missing rainfall history. |
| month_sin | sin(2*pi*(m-1)/12); h is Singapore hour + minute/60, m is month 1-12 | dimensionless | Singapore calendar at prediction time t; no rainfall lookup | not applicable (time only) | Computed for a valid anchor time; independent of missing rainfall history. |
| month_cos | cos(2*pi*(m-1)/12); h is Singapore hour + minute/60, m is month 1-12 | dimensionless | Singapore calendar at prediction time t; no rainfall lookup | not applicable (time only) | Computed for a valid anchor time; independent of missing rainfall history. |
| lag_10m_mm | Rainfall at the exact endpoint t-10min | mm | Endpoint t-10min and its preceding five-minute interval | no | Null if the exact endpoint is absent; do not substitute the preceding row. |
| lag_15m_mm | Rainfall at the exact endpoint t-15min | mm | Endpoint t-15min and its preceding five-minute interval | no | Null if the exact endpoint is absent; do not substitute the preceding row. |
| lag_30m_mm | Rainfall at the exact endpoint t-30min | mm | Endpoint t-30min and its preceding five-minute interval | no | Null if the exact endpoint is absent; do not substitute the preceding row. |
| wet_current | 1 if current_5m_mm>0, otherwise 0 | 0/1 | Endpoint t | yes | Null for a missing lag; genuine observed zero gives 0. |
| wet_lag_5m | 1 if lag_5m_mm>0, otherwise 0 | 0/1 | Exact endpoint t-5min | no | Null for a missing lag; genuine observed zero gives 0. |
| wet_lag_10m | 1 if lag_10m_mm>0, otherwise 0 | 0/1 | Exact endpoint t-10min | no | Null for a missing lag; genuine observed zero gives 0. |
| consecutive_wet_slots_60m | Trailing positive-rainfall slot count from t backwards; 0 when currently dry; capped at 12 | slots | t-55min,...,t | yes | Null for incomplete history; all 12 wet gives 12 with the left-censor flag; this does not establish continuous rain within each slot. |
| wet_run_left_censored_60m | 1 when all 12 slots are wet, indicating the wet run may have started before the observed hour; otherwise 0 | 0/1 | t-55min,...,t | yes | Null unless the required exact historical slots exist; never replace missing rainfall with zero. |
| minutes_since_last_rain_60m_capped | Minutes from t to the latest observed wet-slot endpoint; use code 60 when none is observed | minutes/censoring code | t-55min,...,t; observed ages 0,5,...,55 | yes | Null for incomplete history; complete but all dry gives unknown-gap code 60, paired with no_rain_observed_60m. |
| no_rain_observed_60m | 1 when all 12 observed slots are dry; 0 when at least one is wet | 0/1 | t-55min,...,t | yes | Null for incomplete history; 1 does not imply there has never been rainfall. |
| rain_change_5m_mm | current_5m_mm minus lag_5m_mm | mm | Exact endpoints t-5min and t | yes | Null if either operand is missing. |
| rain_ratio_15m_30m | Past 15-minute rainfall sum divided by past 30-minute sum | dimensionless, 0-1 | t-25min,...,t | yes | All 6 slots required; divide when the denominator is positive; complete all-dry gives 0; missing slots give null. |

## Audit columns outside the default model inputs

n_valid_15m/n_valid_30m/n_valid_60m/n_valid_previous_15m count valid historical positions; history_complete_60m tests all 12. The uncapped minutes_since_last_rain_60m is null when no rain is seen in the hour and is not a default model feature. Prediction timestamps/year are keys or partition fields, not wholesale numeric model inputs. Twelve wet slots do not establish the true storm start or uninterrupted within-slot rainfall. Inference requires complete history and does not use future-label validity.


## Supplement: annual complete-history retention

B retains incomplete histories with nulls; complete-history supervision excludes them. A eligibility also requires future completeness, so this is not final sample retention and overlapping losses cannot be added. Inference rejects gaps.

In [ ]:
show_csv(BROOT / 'results/features_v2/history_retention_by_year.csv', 8)


## Supplement: base join evidence

The displayed audit checks full A/B tables. Actual C filtered/sampled inputs require their own audit before expansion; stop on failures rather than masking duplicates.

In [ ]:
print((BROOT / 'results/join_audit/validation.json').read_text())
show_csv(BROOT / 'results/join_audit/yearly_join_audit.csv', 8)


### Editable one-to-one audit

Left is C actual filtered/sampled anchors; right is B features. Both keys must be unique and every left must match once. Unused right anchors are permitted. Expanded keys additionally include threshold.

In [ ]:
KEYS = ['station_id', 'prediction_epoch_s']

def audit_anchor_join(left,right,scope='C_pre_expansion',require_all_left=True,require_all_right=False):
    """left = C supervised anchors; right = B features. Invoke on actual C inputs.

    Audit precedes threshold expansion: repeated anchor/threshold rows are not valid inputs here.
    Returns evidence; caller performs the join only after PASS. No rain/target values are collected.
    """
    from pyspark.sql import functions as F
    def describe(frame):
        keys=frame.select(*KEYS)
        nulls=keys.filter(F.col(KEYS[0]).isNull()|F.col(KEYS[1]).isNull()).count()
        counts=keys.groupBy(*KEYS).count()
        info=counts.agg(F.count('*').alias('unique_keys'),F.sum('count').alias('rows'),
            F.sum(F.when(F.col('count')>1,F.lit(1)).otherwise(0)).alias('duplicate_key_groups'),
            F.sum(F.col('count')-1).alias('duplicate_excess_rows')).first().asDict()
        return keys,{k:int(v or 0) for k,v in info.items()}|{'null_key_rows':nulls}
    lk,ls=describe(left);rk,rs=describe(right)
    result={'scope':scope,**{'left_'+k:v for k,v in ls.items()},**{'right_'+k:v for k,v in rs.items()}}
    clean=ls['null_key_rows']==rs['null_key_rows']==ls['duplicate_excess_rows']==rs['duplicate_excess_rows']==0
    if not clean:
        result.update(status='FAIL',reason='Null or duplicate keys; join was not executed')
        return result
    unmatched_left=lk.join(rk,KEYS,'left_anti').count()
    unmatched_right=rk.join(lk,KEYS,'left_anti').count()
    inner_rows=lk.join(rk,KEYS,'inner').count()
    left_join_rows=lk.join(rk,KEYS,'left').count()
    result.update(left_unmatched_rows=unmatched_left,right_unmatched_rows=unmatched_right,
        matched_left_rows=ls['rows']-unmatched_left,inner_join_rows=inner_rows,left_join_rows=left_join_rows,
        left_match_rate=(ls['rows']-unmatched_left)/ls['rows'] if ls['rows'] else None,
        right_match_rate=(rs['rows']-unmatched_right)/rs['rows'] if rs['rows'] else None,
        row_inflation=left_join_rows-ls['rows'])
    ok=(inner_rows==ls['rows']-unmatched_left and left_join_rows==ls['rows']
        and (not require_all_left or unmatched_left==0) and (not require_all_right or unmatched_right==0))
    result['status']='PASS' if ok else 'FAIL'
    return result


### Example C actual-input call (disabled by default)

Provide real C_ANCHORS/B_FEATURES pre-expansion DataFrames. This disabled example is not evidence of completed C acceptance; actual C results are recorded separately.

In [ ]:
RUN_C_ACTUAL_JOIN_AUDIT = False
if RUN_C_ACTUAL_JOIN_AUDIT:
    assert 'C_ANCHORS' in globals() and 'B_FEATURES' in globals(), 'Provide actual C pre-expansion anchors and B feature DataFrames first.'
    check = audit_anchor_join(C_ANCHORS, B_FEATURES, scope='C_actual_pre_threshold_expansion')
    dump(BROOT / 'results/join_audit/C_actual_input_validation.json', check)
    assert check['status'] == 'PASS', check
else:
    print('This example actual-input audit is disabled; base-table audit passed. Completed C run checks are saved separately.')


## Environment scope

Hardware was inventoried after runs; peak memory was not measured. Original timing includes reads/writes but excludes later reports/figures/hashes/shutdown and separately added checks. See the English B report.

In [ ]:
print((BROOT / 'results/hardware_inventory.json').read_text())
